# Quality Assessment
Now it's your turn. Follow the steps on the platform and use what you've learnt to see how reliable the data is.

In [2]:
import pandas as pd

Load our cleaned DataFrames

In [3]:
# orders_cl.csv
url = "https://drive.google.com/file/d/1Tla62vfu__kCqvgypZyVt2S9VuC016yH/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders_cl = pd.read_csv(path)

# orderlines_cl.csv
url = "https://drive.google.com/file/d/1OhtkQS2fwOYdzfd-qPh7im35iLc-L9TA/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines_cl = pd.read_csv(path)

# products_cl.csv
url = "https://drive.google.com/file/d/1s7Lai4NSlsYjGEPg1QSOUJobNYVsZBOJ/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products_cl = pd.read_csv(path)

## 1.&nbsp; Define Pandas display format

In [15]:
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)


## 2.&nbsp; Exclude unwanted orders

In [16]:
valid_states = ["Completed", "Shipped", "Delivered"]

orders_cl = orders_cl[orders_cl["state"].isin(valid_states)]


## 3.&nbsp; Exclude orders with unknown products


In [17]:
valid_skus = products_cl["sku"].unique()

orderlines_cl = orderlines_cl[orderlines_cl["sku"].isin(valid_skus)]


## 4.&nbsp; Explore the revenue from different tables

#### Step 1:
Create the `unit_price_total` as `orderlines.unit_price` * `orderlines.product_quantity`

In [18]:
orderlines_cl["unit_price_total"] = (
    orderlines_cl["unit_price"] * orderlines_cl["product_quantity"]
)


#### Step 2:
Group by `id_order`, summarising by the sum of `unit_price_total`

In [20]:
order_totals = (
    orderlines_cl
    .groupby("id_order")["unit_price_total"]
    .sum()
    .reset_index()
)


### What is the average difference between `total_paid` and `unit_price_total`?

In [21]:
comparison = order_totals.merge(
    orders_cl,
    left_on="id_order",
    right_on="order_id",
    how="inner"
)


In [22]:
comparison["difference"] = (
    comparison["total_paid"] - comparison["unit_price_total"]
)


In [23]:
comparison["difference"].mean()


np.float64(5.934310797699898)

### What is the distribution of these differences?

In [24]:
comparison["difference"].describe()
comparison["difference"].value_counts().head(10)


,count
difference,
0.00,10484
6.99,4559
4.99,3419
4.99,2523
4.99,2352
6.99,1890
3.99,1882
3.99,1620
3.99,1463


### Can all the differences be explained by shipping costs? If not, what are other plausible explanations?

No.

Shipping is usually a small, fairly constant amount (e.g., €5–€10).
But here we observe that the differences:

are sometimes large

can be negative

can vary widely

Other plausible explanations include:

Discounts applied to the whole order (not per item)

Coupons or promotions

Rounding errors

Partial order cancellations or refunds

Incorrect prices recorded in orderlines

VAT/taxes included in one table but not the other

What should we do with the unexplained differences?

As a careful data analyst, we keep only the orders where the difference is negligible

### If there are differences that you can’t explain: what should you do with these orders?

In [25]:
clean_orders = comparison[comparison["difference"].abs() < 1]


## 5.&nbsp; Become confident about your dataset

Before proceeding to exploratory data analysis and visualization, the consistency of the dataset was carefully validated. By comparing the revenue calculated from orderlines with the total_paid values from the orders table, noticeable differences were identified and investigated. Since many of these differences could not be fully explained by shipping costs alone, orders with significant mismatches were filtered out. This step ensured that only reliable and consistent orders remained in the dataset. As a result, the revenue data can now be trusted, and the dataset is ready for further analysis and visualization.